# M5 Agentic AI - 市場調研團隊

## 1. 引言

### 1.1. 實驗概覽  

在本實驗中，你將扮演一家時尚品牌的**技術 AI 負責人**，籌備夏季太陽鏡活動。你的任務是設計一條**全自動的創意流水線**，貼近真實業務場景。與其逐個手動處理，你將引導一個系統：掃描線上來源以發現新興的時尚趨勢，將這些趨勢與內部目錄中的太陽鏡匹配，設計活動視覺，生成簡短營銷文案，並最終將所有內容打包成一份**可供高管審閱的報告**。  

目標是體驗如何將多個代理、工具與模型編排為一個統一、連貫的工作流。完成本實驗後，你將構建出一個更像小團隊協作解決創意挑戰的流水線，而非一串相互孤立的指令碼步驟。  

### 1.2. 🎯 學習目標

透過本實驗，你將看到如何超越模型的單輪互動，轉而設計**多代理流水線**來協同規劃、調研與創意生成。你將學習如何用外部工具支撐代理的推理，使輸出不僅富有創意，也有真實資料背書。你還會嘗試反思與打包步驟，以實施質量控制並為高管受眾準備結果。  

簡而言之，本實驗旨在學習如何將**大語言模型的想象力**與**結構化工作流的紀律性**結合起來，為構建既有創意又可靠的自治系統提供實用正規化。  

## 2. 環境設定：匯入庫並載入環境

與之前的實驗一樣，現在匯入所需庫、載入環境變數並設定輔助工具。

In [ ]:
# =========================
# 匯入
# =========================

# --- 標準庫 ---
import base64
import json
import os
import re
from datetime import datetime
from io import BytesIO

# --- 第三方 ---
import requests
import openai
from PIL import Image
from dotenv import load_dotenv
from IPython.display import Markdown, display
import aisuite

# --- 本地 / 專案 ---
import tools
import utils


# =========================
# 環境與客戶端
# =========================
load_dotenv()
client = aisuite.Client()


## 3. 可用工具  

只有當模型在基礎推理之外被賦予**明確能力**時，自治流水線才真正奏效。預先宣告這些工具可使代理的動作空間清晰明確、讓提示自然引導工具選擇，並透過定義良好的介面保持編排與測試的透明性。  

你將組建一個**市場調研團隊**，由多個專業代理協作設計夏季太陽鏡活動。為賦能他們，我們首先定義能以真實資料支撐其推理的工具。  

第一個工具是 `tools.tavily_search_tool`，用於進行實時網頁搜尋以挖掘當前時尚趨勢的證據。現在試試，執行一個簡單查詢：*“trends in sunglasses fashion”*。  

In [ ]:
tools.tavily_search_tool('trends in sunglasses fashion')

第二個工具是 `tools.product_catalog_tool`，它返回內部的太陽鏡目錄。每條記錄都包含產品名稱、ID、描述、庫存數量與價格等詳情。這些結構化資料使智慧體能夠將線上時尚趨勢與實際庫存商品關聯起來：

In [ ]:
tools.product_catalog_tool()

有了這些工具，你已定義了清晰的動作空間與可靠的資料來源。下一節將構建使用這些工具的代理，把原始的時尚訊號轉化為結構化洞見與活動資產。

## 4. 智慧體定義 — 組建你的團隊

既然工具已就位，現在讓它們發揮作用。在這一階段，你將組建一個**市場調研團隊**，由多名專業代理構成，並用自然語言指令進行驅動。  

每個代理都依賴此前引入的工具，協同把原始趨勢資料轉化為打磨完善的活動報告。我們將逐個定義它們，介紹其角色並展示對應實現程式碼。  

### 4.1. 市場調研智慧體  

藉助**市場調研智慧體**，你邁出構建活動的第一步。讓它使用 `tavily_search_tool` 掃描網路，洞察當前太陽鏡時尚的熱點趨勢；隨後用 `product_catalog_tool` 將這些訊號與內部目錄交叉驗證，從而知道哪些產品契合當下。  

該智慧體會返回一份簡潔的簡報：所發現的頭部時尚洞見、與之匹配的產品，以及為何這些選擇適合你的夏季推廣的簡短說明。這為後續活動的塑造提供了清晰、資料驅動的基礎。  

現在可以執行下方單元，以程式碼形式定義**市場調研智慧體**。  

In [ ]:
def market_research_agent(return_messages: bool = False):

    utils.log_agent_title_html("Market Research Agent", "🕵️‍♂️")

    prompt_ = f"""
你是一名時尚市場調研代理，負責為夏季太陽鏡活動準備趨勢分析。

目標：
1. 使用網頁搜尋探索與太陽鏡相關的當前時尚趨勢。
2. 檢視內部產品目錄，識別與這些趨勢相契合的商品。
3. 從目錄中推薦一個或多個最符合新興趨勢的產品。
4. 如需註明，今天的日期是 {datetime.now().strftime("%Y-%m-%d")}。

可呼叫以下工具：
- tavily_search_tool：發現外部網路趨勢。
- product_catalog_tool：檢查內部太陽鏡目錄。

完成分析後，請總結：
- 你發現的 2–3 個主要趨勢。
- 與這些趨勢匹配的目錄產品。
- 為何它們適合夏季活動的理由說明。
"""
    messages = [{"role": "user", "content": prompt_}]
    tools_ = tools.get_available_tools()

    while True:
        response = client.chat.completions.create(
            model="openai:o4-mini",
            messages=messages,
            tools=tools_,
            tool_choice="auto"
        )

        msg = response.choices[0].message

        if msg.content:
            utils.log_final_summary_html(msg.content)
            return (msg.content, messages) if return_messages else msg.content

        if msg.tool_calls:
            for tool_call in msg.tool_calls:
                utils.log_tool_call_html(tool_call.function.name, tool_call.function.arguments)
                result = tools.handle_tool_call(tool_call)
                utils.log_tool_result_html(result)

                messages.append(msg)
                messages.append(tools.create_tool_response_message(tool_call, result))
        else:
            utils.log_unexpected_html()
            return ("[⚠️ Unexpected: No tool_calls or content returned]", messages) if return_messages else "[⚠️ Unexpected: No tool_calls or content returned]"

讓我們從**市場調研代理**獲取關於夏季太陽鏡活動的建議。  

In [ ]:
market_research_result = market_research_agent()

接下來，使用平面設計代理將這份簡報轉化為視覺概念。

### 4.2. 平面設計智慧體  

藉助**平面設計智慧體**，你將從分析走向創意。  
獲取市場調研智慧體的簡報，並讓該智慧體將其轉化為視覺概念。  
由於 `aisuite` 尚不支援直接影象生成（如 DALL·E），你將以兩階段引導流程：  

1. 首先，代理使用 `aisuite` 與 OpenAI 文字模型（`o4-mini`）生成生動的**提示**與簡短、吸引人的**文案**。  
2. 然後，將該提示傳送到 OpenAI 的 `dall-e-3` API 以生成**活動影象**本身。  

結果提供了推進所需的一切：生成的影象（本地儲存以便複用）、生成它的精確提示（便於迭代）、以及用於活動敘事的精煉文案。  

<div style="border:1px solid #fca5a5; border-left:6px solid #ef4444; background:#fee2e2; border-radius:6px; padding:12px 14px; color:#111827; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">
  <strong>注意：</strong> 當前 <code>aisuite</code> <strong>不支援直接影象生成</strong>。  
  因此你將其文字輸出（提示 + 文案）與 OpenAI 的 <code>dall-e-3</code> 結合，用於生成最終的活動視覺。
</div>  

現在可以執行下方單元，以程式碼形式定義**平面設計代理**。  

In [ ]:
def graphic_designer_agent(trend_insights: str, caption_style: str = "short punchy", size: str = "1024x1024") -> dict:

    """
    使用 aisuite 生成營銷提示/文案，並直接使用 OpenAI 生成影象。

    引數：
        trend_insights (str)：來自調研智慧體的趨勢摘要。
        caption_style (str)：文案的可選風格提示。
        size (str)：影象解析度（例如 '1024x1024'）。

    返回：
        dict：包含 image_url、prompt 與 caption 的字典。
    """

    utils.log_agent_title_html("平面設計智慧體", "🎨")

    # 步驟 1: 使用 aisuite 生成提示和文案
    system_message = (
        "你是一名視覺營銷助理。根據輸入的趨勢洞見，"
        "為 AI 影象生成模型編寫一個有創意的視覺提示，並生成一段簡短文案。"
    )

    user_prompt = f"""
趨勢洞見：
{trend_insights}

請輸出：
1. 一段生動、具描述性的提示，用於引導影象生成。
2. 一句營銷文案，風格：{caption_style}。

按如下格式回應：
{{"prompt": "...", "caption": "..."}}
"""

    chat_response = client.chat.completions.create(
        model="openai:o4-mini",
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_prompt}
        ]
    )

    content = chat_response.choices[0].message.content.strip()
    match = re.search(r'\{.*\}', content, re.DOTALL)
    parsed = json.loads(match.group(0)) if match else {"error": "No JSON returned", "raw": content}

    prompt = parsed["prompt"]
    caption = parsed["caption"]

    # 步驟 2: 直接使用 openai-python 生成影象
    openai_client = openai.OpenAI()

    image_response = openai_client.images.generate(
        model="dall-e-3",
        prompt=prompt,
        size=size,
        quality="standard",
        n=1,
        response_format="url"
    )

    image_url = image_response.data[0].url

    # 本地儲存影象
    img_bytes = requests.get(image_url).content
    img = Image.open(BytesIO(img_bytes))

    filename = os.path.basename(image_url.split("?")[0])
    image_path = filename
    img.save(image_path)


    # 使用本地影象記錄摘要
    utils.log_final_summary_html(f"""
        <h3>已生成影象與文案</h3>

        <p><strong>影象路徑：</strong> <code>{image_path}</code></p>

        <p><strong>生成的影象：</strong></p>
        <img src="{image_path}" alt="Generated Image" style="max-width: 100%; height: auto; border: 1px solid #ccc; border-radius: 8px; margin-top: 10px; margin-bottom: 10px;">

        <p><strong>提示：</strong> {prompt}</p>
    """)


    return {
        "image_url": image_url,
        "prompt": prompt,
        "caption": caption,
        "image_path": image_path  
    }



現在執行 `graphic_designer_agent`，使用**市場調研智慧體**提供的趨勢洞見生成活動影象。

In [ ]:
graphic_designer_agent_result = graphic_designer_agent(
    trend_insights=market_research_result,
)


拿到視覺素材後，使用文案智慧體來打造活動的聲音。

### 4.3. 文案智慧體  

在**市場調研智慧體**與**平面設計智慧體**完成其工作後，轉向**文案智慧體**。當你手頭已有活動影象與趨勢摘要時，讓該智慧體為活動建立“聲音”。  

它將視覺與分析作為多模態輸入，打造一條簡短而優雅的營銷短句，精準傳達核心資訊。除了短句，它還提供清晰的理由——為何該短句契合影象，以及它如何與趨勢相呼應。  

這樣，你不僅獲得朗朗上口的文案，也得到其背後的推理，便於在利益相關者面前闡述與最佳化。  



In [ ]:
def copywriter_agent(image_path: str, trend_summary: str, model: str = "openai:o4-mini") -> dict:

    """
    使用 aisuite（僅 OpenAI）傳送影象與趨勢摘要並返回活動短句。

    引數：
        image_path (str)：待分析影象的路徑。
        trend_summary (str)：來自調研智慧體的文字。
        model (str)：OpenAI 模型（例如 openai:o4-mini、openai:gpt-4o）

    返回：
        dict: {
            "quote": "...",
            "justification": "...",
            "image_path": "..."
        }
    """

    utils.log_agent_title_html("文案智慧體", "✍️")

    # 步驟 1: 載入本地影象並編碼為 base64
    with open(image_path, "rb") as f:
        img_bytes = f.read()

    b64_img = base64.b64encode(img_bytes).decode("utf-8")

    # 步驟 2: 構建相容 OpenAI 的多模態訊息
    messages = [
        {
            "role": "system",
            "content": "你是一名文案撰寫者，基於影象與市場趨勢摘要創作優雅的活動短句。"
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{b64_img}",
                        "detail": "auto"
                    }
                },
                {
                    "type": "text",
                    "text": f"""
以下為一個視覺營銷影象與趨勢分析：

趨勢摘要：
\"\"\"{trend_summary}\"\"\"

請返回如下 JSON 物件：
{{
  "quote": "簡短、優雅的活動短句（最多 12 個詞）",
  "justification": "為何該短句契合該影象與趨勢"
}}"""
                }
            ]
        }
    ]

    # 步驟 3: 透過 aisuite 傳送請求
    response = client.chat.completions.create(
        model=model,
        messages=messages,
    )

    # 步驟 4: 解析 JSON 響應
    content = response.choices[0].message.content.strip()

    utils.log_final_summary_html(content)

    try:
        match = re.search(r'\{.*\}', content, re.DOTALL)
        parsed = json.loads(match.group(0)) if match else {"error": "No valid JSON returned"}
    except Exception as e:
        parsed = {"error": f"Failed to parse: {e}", "raw": content}


    parsed["image_path"] = image_path
    return parsed


現在呼叫文案智慧體，根據營銷影象與先前生成的趨勢洞見，產出一條簡短的活動短句。

In [ ]:
copywriter_agent_result = copywriter_agent(
    image_path=graphic_designer_agent_result["image_path"],
    trend_summary=market_research_result,
)

當短句與理由準備好後，使用打包智慧體將所有內容整合為可供高管審閱的成果。

### 4.4. 打包智慧體  

最後，引入**打包智慧體**將所有內容串聯成整體。在**市場調研智慧體**、**平面設計智慧體**與**文案智慧體**各自完成其部分後，打包智慧體會把整個故事整合為一個打磨完善的成果。  

讓它接收趨勢摘要、活動視覺、生成的短句與理由說明，並將這些內容組裝成一份適合高管閱讀的 Markdown 報告。過程中，它會重寫趨勢洞見以提升清晰度與語氣、確保短句與影象樣式搭配合理，並組織結構使最終文件專業且具說服力。  

完成此步後，你將獲得一套完整的活動資料包——易於分享、視覺吸引、並準備好接受高層審閱。  

In [ ]:
def packaging_agent(trend_summary: str, image_url: str, quote: str, justification: str, output_path: str = "campaign_summary.md") -> str:

    """
    將活動資產打包為精美的 Markdown 報告，供高管審閱。

    Args:
        trend_summary (str)：市場趨勢摘要。
        image_url (str)：活動影象的 URL。
        quote (str)：需疊加的營銷短句。
        justification (str)：短句的理由說明。
        output_path (str)：儲存 Markdown 報告的路徑。

    Returns:
        str：已儲存的 Markdown 檔案路徑。
    """

    utils.log_agent_title_html("打包智慧體", "📦")

    # 我們在 <img> 的 src 中使用此路徑
    styled_image_html = f"""
![開啟生成的檔案檢視]({image_url})
    """

    beautified_summary = client.chat.completions.create(
        model="openai:o4-mini",
        messages=[
            {"role": "system", "content": "你是一名市場傳播專家，為高管撰寫優雅的活動總結。"},
            {"role": "user", "content": f"""
請將以下趨勢摘要改寫為清晰、專業且適合 CEO 受眾的表達：

\"\"\"{trend_summary.strip()}\"\"\"
"""}
        ]
    ).choices[0].message.content.strip()

    utils.log_tool_result_html(beautified_summary)

    # 將所有部分合並為 markdown
    markdown_content = f"""# 🕶️ 夏季太陽鏡活動 – 高管摘要

## 📊 精煉的趨勢洞見
{beautified_summary}

## 🎯 活動視覺
{styled_image_html}

## ✍️ 活動短句
{quote.strip()}

## ✅ 原因說明
{justification.strip()}

---

*報告生成日期 {datetime.now().strftime('%Y-%m-%d')}*
"""

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(markdown_content)

    return output_path



當趨勢摘要、活動影象與短句都已就緒後，將所有內容交給**打包智慧體**。它的職責是把這些部分整合為一份精緻、可供高管審閱的報告。執行下一個單元來生成它。  


In [ ]:
packaging_agent_result = packaging_agent(
    trend_summary=market_research_result,
    image_url=graphic_designer_agent_result["image_path"],
    quote=copywriter_agent_result["quote"],
    justification=copywriter_agent_result["justification"],
    output_path=f"campaign_summary_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.md"
)

最終結果是一份格式精美的活動報告，你可以在筆記本中直接檢視。它將包含：  

- 經精煉的趨勢摘要，改寫為更適合高管閱讀的表述  
- 視覺化樣式的影象，並透過 HTML 疊加你的活動短句  
- 清晰的理由說明，幫助你理解為何視覺與資訊契合當前趨勢  
- 顯示報告生成時間的時間戳  

你可以使用如下方式檢視：  

In [ ]:
# 載入並渲染 Markdown 內容
with open(packaging_agent_result, "r", encoding="utf-8") as f:
    md_content = f.read()

display(Markdown(md_content))


最後，將整個工作流封裝為一個可呼叫函式，以一步執行完整流水線。

## 5. 完整活動流水線 – `run_sunglasses_campaign_pipeline`

在此步驟中，你將定義一個函式 `run_sunglasses_campaign_pipeline`，把所有部分串聯為一個無縫的夏季太陽鏡活動工作流。  

該函式將：  
- 執行市場調研，掃描時尚趨勢並與目錄匹配。  
- 生成具有視覺樣式的影象與文案。  
- 建立簡短、優雅且帶理由說明的活動短句。  
- 將所有內容打包為精美的 Markdown 報告，便於高管審閱。  

定義該函式後，你可以**一次呼叫執行整個流水線**，同時仍能追蹤中間結果並檢視最終報告。  

In [ ]:
def run_sunglasses_campaign_pipeline(output_path: str = "campaign_summary.md") -> dict:
    """
    執行完整的夏季太陽鏡活動流水線：
    1. 市場調研（搜尋趨勢並匹配產品）
    2. 生成視覺影象與文案
    3. 基於影象與趨勢生成活動短句
    4. 建立高管版 Markdown 報告

    Returns:
        dict: 包含所有中間結果與最終報告路徑的字典
    """
    # 1. 執行市場調研Agent
    trend_summary = market_research_agent()
    print("✅ 市場調研完成")

    # 2. 生成影象 + 文案
    visual_result = graphic_designer_agent(trend_insights=trend_summary)
    image_path = visual_result["image_path"]
    print("🖼️ 影象已生成")

    # 3. 基於影象 + 趨勢生成短句
    quote_result = copywriter_agent(image_path=image_path, trend_summary=trend_summary)
    quote = quote_result.get("quote", "")
    justification = quote_result.get("justification", "")
    print("💬 短句已生成")

    # 4. 生成 markdown 報告
    md_path = packaging_agent(
        trend_summary=trend_summary,
        image_url=image_path,  
        quote=quote,
        justification=justification,
        output_path=f"campaign_summary_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.md"
    )

    print(f"📦 報告已生成：{md_path}")

    return {
        "trend_summary": trend_summary,
        "visual": visual_result,
        "quote": quote_result,
        "markdown_path": md_path
    }


現在可透過一次呼叫執行流水線來建立完整的活動報告。執行下一個單元即可：  

In [ ]:
results = run_sunglasses_campaign_pipeline()

### 5.1. 結果

執行下方單元以檢視完整活動流水線生成的輸出。

In [ ]:
with open(results["markdown_path"], "r", encoding="utf-8") as f:
    md_content = f.read()
display(Markdown(md_content))


## 6. 關鍵要點  

完成本實驗後，你已瞭解如何：  

- 使用**多代理 LLM 流水線**端到端地自動化創意工作流。  
- 將**推理、工具呼叫與外部資料**結合，使輸出紮根於現實。  
- 應用多模態模型（如 `gpt-4o`）處理**文字與影象**，用於生成活動短句等任務。  
- 透過工具（`tavily_search_tool`、`product_catalog_tool`）擴充套件模型能力，使輸出不僅富有創意，也更務實。  
- 藉助結構化日誌與 HTML 樣式塊保持執行**透明且可除錯**。  
- 交付格式精美、**適合高管審閱**的 Markdown 報告，將洞見、視覺與理由整合為一個成果。  



<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

🎉 <strong>恭喜！</strong> 🎉  

你已成功構建並執行一條**多代理流水線**：完成趨勢調研、生成視覺素材、打造活動短句，並把一切打包為**適合高管審閱的報告**。  

該工作流展示瞭如何將 **LLM 的創造力** 與 **結構化編排的紀律性**結合，為你提供可複用的正規化，能夠適配多種現實場景。🌟  
</div>
